# Fabric Canonical Model: pruebas de integración Delta

Prueba de integración manual para ejecutar en Microsoft Fabric. Valida persistencia `FULL_LOAD`, `UPSERT`, `SCD2`, continuidad de `Id`, watermarks por fuente, auditoría y que un callback fallido no confirme su watermark.

**Escrituras:** crea un schema de prueba único y tablas de control/auditoría con una entidad de prueba única. No borra automáticamente los resultados. Revise las tablas creadas al final y elimínelas manualmente cuando ya no las necesite. Requiere el wheel en el Environment y un Lakehouse adjunto con Delta habilitado.

In [ ]:
from datetime import datetime
from uuid import uuid4

from fabric_canonical_model.exceptions import ExecutionError, PersistenceError
from fabric_canonical_model.lib_audit import ExecutionAudit
from fabric_canonical_model.lib_execution import EntityExecutionManager
from fabric_canonical_model.lib_persistence import CanonicalPersistence
from fabric_canonical_model.lib_utils import get_spark
from fabric_canonical_model.lib_watermark import WatermarkManager

spark = get_spark()
run_suffix = uuid4().hex[:10]
test_entity = f"__framework_test_{run_suffix}"
test_schema = f"canonical_test_{run_suffix}"
audit_table = f"execution_audit_test_{run_suffix}"
persistence = CanonicalPersistence(spark=spark, target_schema=test_schema)
watermarks = WatermarkManager(spark=spark)
audit = ExecutionAudit(
    spark=spark,
    schema_name="canonical_metadata",
    table_name=audit_table
)

def frame(rows):
    return spark.createDataFrame(rows, ["UWI", "Well_Name"])

def table(table_name):
    return spark.table(f"{test_schema}.{table_name}")

def by_key(dataframe):
    return {row["UWI"]: row.asDict(recursive=True) for row in dataframe.collect()}

## UPSERT: claves sustitutas y auditoría funcional

Comprueba inserciones, actualización de atributos, preservación de `Id`/`fecha_creacion` y que una repetición sin cambios no altere `fecha_actualizacion`.

In [ ]:
persistence.upsert(
    frame([("U-1", "Pozo Uno"), ("U-2", "Pozo Dos")]),
    "test_upsert",
    ["UWI"]
)
initial = by_key(table("test_upsert"))

persistence.upsert(
    frame([("U-1", "Pozo Uno"), ("U-2", "Pozo Dos")]),
    "test_upsert",
    ["UWI"]
)
unchanged = by_key(table("test_upsert"))
assert unchanged["U-1"]["Id"] == initial["U-1"]["Id"]
assert unchanged["U-1"]["fecha_creacion"] == initial["U-1"]["fecha_creacion"]
assert unchanged["U-1"]["fecha_actualizacion"] == initial["U-1"]["fecha_actualizacion"]

persistence.upsert(
    frame([("U-1", "Pozo Uno Actualizado"), ("U-2", "Pozo Dos"), ("U-3", "Pozo Tres")]),
    "test_upsert",
    ["UWI"]
)
updated = by_key(table("test_upsert"))
assert len(updated) == 3
assert updated["U-1"]["Id"] == initial["U-1"]["Id"]
assert updated["U-1"]["fecha_creacion"] == initial["U-1"]["fecha_creacion"]
assert updated["U-1"]["Well_Name"] == "Pozo Uno Actualizado"
print("UPSERT: OK")

## FULL_LOAD: reemplazo conservando identidad

El snapshot reemplaza las filas ausentes, pero conserva el `Id` de claves existentes.

In [ ]:
persistence.full_load(
    frame([("F-1", "Campo Uno"), ("F-2", "Campo Dos")]),
    "test_full_load",
    ["UWI"]
)
full_initial = by_key(table("test_full_load"))
persistence.full_load(
    frame([("F-1", "Campo Uno Revisado"), ("F-3", "Campo Tres")]),
    "test_full_load",
    ["UWI"]
)
full_after = by_key(table("test_full_load"))
assert set(full_after) == {"F-1", "F-3"}
assert full_after["F-1"]["Id"] == full_initial["F-1"]["Id"]
assert full_after["F-1"]["fecha_creacion"] == full_initial["F-1"]["fecha_creacion"]
print("FULL_LOAD: OK")

## SCD2: versiones históricas

Valida una versión vigente por clave, creación de historial ante cambio y ausencia de una versión adicional cuando el contenido no cambia. En la implementación actual, el `Id` identifica la entidad y se conserva entre versiones.

In [ ]:
persistence.scd2(
    frame([("S-1", "Yacimiento Uno")]),
    "test_scd2",
    ["UWI"]
)
scd_initial = table("test_scd2").collect()[0].asDict(recursive=True)
persistence.scd2(
    frame([("S-1", "Yacimiento Uno Actualizado")]),
    "test_scd2",
    ["UWI"]
)
versions = table("test_scd2").orderBy("fecha_inicio_vigencia").collect()
assert len(versions) == 2
assert sum(1 for row in versions if row["registro_actual"]) == 1
assert all(row["Id"] == scd_initial["Id"] for row in versions)
persistence.scd2(
    frame([("S-1", "Yacimiento Uno Actualizado")]),
    "test_scd2",
    ["UWI"]
)
assert table("test_scd2").count() == 2
print("SCD2: OK")

## Watermarks por fuente y fallo antes de confirmar

El callback de prueba persiste primero y entrega un watermark candidato. Después se comprueba que la ejecución fallida no invoque `commit_watermark`.

In [ ]:
avm_candidate = datetime(2026, 11, 23, 12, 0, 0)
bdp_candidate = datetime(2026, 11, 23, 13, 0, 0)
execution_manager = EntityExecutionManager(audit, watermarks)

def persist_then_return_candidate(execution_id):
    persistence.upsert(
        frame([("E-1", "Execution test")]),
        "test_execution",
        ["UWI"]
    )
    return {
        "metrics": {"records_read": 1},
        "candidate_watermarks": {"AVM": avm_candidate, "BDP": bdp_candidate}
    }

success = execution_manager.execute_entity(
    entity_name=test_entity,
    execution_mode="UPSERT",
    source_names=["AVM", "BDP"],
    process_function=persist_then_return_candidate,
    process_name="nb_fabric_integration_tests"
)
assert success["execution_status"] == "SUCCESS"
assert watermarks.get_watermark(test_entity, "AVM") == avm_candidate
assert watermarks.get_watermark(test_entity, "BDP") == bdp_candidate

before_failed_run = watermarks.get_watermark(test_entity, "AVM")
def fail_before_persist(execution_id):
    raise RuntimeError("fallo simulado antes de persistir")

try:
    execution_manager.execute_entity(
        entity_name=test_entity,
        execution_mode="UPSERT",
        source_names=["AVM"],
        process_function=fail_before_persist,
        process_name="nb_fabric_integration_tests"
    )
except ExecutionError:
    pass
else:
    raise AssertionError("La ejecución fallida no propagó el error")
assert watermarks.get_watermark(test_entity, "AVM") == before_failed_run
assert audit.get_entity_history(test_entity, limit=10).count() == 2
print("Watermarks y auditoría: OK")

## Resultado y limpieza

Si todas las aserciones pasaron, la última celda muestra `ALL INTEGRATION TESTS PASSED`. Las tablas de salida quedan en el schema único impreso abajo; auditoría y watermarks quedan en `canonical_metadata`, identificables por `test_entity`.

La prueba simula una falla **antes** de persistir. No demuestra atomicidad ante una caída entre una escritura Delta exitosa y la confirmación del watermark, concurrencia ni recuperación tras reinicio.

In [ ]:
print("ALL INTEGRATION TESTS PASSED")
print("Test entity:", test_entity)
print("Test schema:", test_schema)
print("Audit table:", f"canonical_metadata.{audit_table}")
print("Watermark table:", watermarks.table_name)